# Per-Store LightGBM Forecasting with MLflow Tracking

Trains a store-level LightGBM model on individual store-day rows (vs. the chain-wide aggregation in notebook 03).
The trained model powers the `POST /forecast/store` and `POST /forecast/store/whatif` API endpoints.

**Key differences from notebook 03 (chain-wide model):**
- Data: individual store-day rows, not daily aggregates across all 1,115 stores
- Features: includes `Store` ID + store metadata from `store.csv` (StoreType, Assortment, CompetitionDistance, Promo2)
- Lag features: computed per-store using `groupby('Store').shift()`
- Target: individual store daily sales (~0-40,000 range)

**MLflow experiment tracking:**
Four runs are logged — baseline (deployed) + three hyperparameter variants:
- `lightgbm_store_baseline` — n_estimators=500, lr=0.05, num_leaves=31 (deployed)
- `lightgbm_store_more_leaves` — num_leaves=63
- `lightgbm_store_lower_lr` — n_estimators=700, lr=0.02
- `lightgbm_store_deeper` — max_depth=8

Run `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root to compare all runs.

## 1. Imports & Constants

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

ROOT       = Path("..").resolve()
TRAIN_CSV  = ROOT / "Data" / "raw" / "train.csv"
STORE_CSV  = ROOT / "Data" / "raw" / "store.csv"
MODEL_OUT  = ROOT / "models" / "lightgbm_store_model.pkl"
FEAT_OUT   = ROOT / "models" / "store_features.pkl"
FIG_DIR    = ROOT / "figures" / "03_forecasting_modern"

SPLIT_DATE = "2015-06-01"

FEATURES = [
    "Store", "DayOfWeek", "Month", "Year", "DayOfMonth",
    "Promo", "SchoolHoliday", "StateHoliday",
    "StoreType", "Assortment", "CompetitionDistance", "Promo2",
    "Sales_Lag_1", "Sales_Lag_7", "Sales_Lag_14",
    "Sales_Rolling_Mean_7", "Sales_Rolling_Mean_14",
]

# SQLite backend -- same db as notebook 04 so all runs appear in one experiment
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("rossmann-demand-forecasting")
print("MLflow ready.")

## 2. Data Loading

In [ ]:
train = pd.read_csv(TRAIN_CSV, low_memory=False)
store = pd.read_csv(STORE_CSV)
print(f"train shape : {train.shape}")
print(f"store shape : {store.shape}")
train.head(3)

## 3. Encode Store Metadata

In [ ]:
def encode_store_meta(store_df: pd.DataFrame) -> pd.DataFrame:
    s = store_df.copy()
    s["StoreType"]  = s["StoreType"].str.lower().map({"a":1,"b":2,"c":3,"d":4}).fillna(0).astype(int)
    s["Assortment"] = s["Assortment"].str.lower().map({"a":1,"b":2,"c":3}).fillna(0).astype(int)
    s["CompetitionDistance"] = s["CompetitionDistance"].fillna(s["CompetitionDistance"].median())
    return s[["Store","StoreType","Assortment","CompetitionDistance","Promo2"]].set_index("Store")

store_meta_idx = encode_store_meta(store)

joblib.dump(store_meta_idx.to_dict(orient="index"), FEAT_OUT)
print(f"Store features saved -> {FEAT_OUT}")
store_meta_idx.head()

## 4. Feature Engineering

In [ ]:
df = train.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df[df["Open"] == 1].copy()
print(f"Rows after filtering closed days: {len(df):,}")

df["StateHoliday"] = df["StateHoliday"].apply(lambda x: 0 if (x=="0" or x==0) else 1).astype(int)
df["DayOfWeek"]  = df["Date"].dt.dayofweek + 1
df["Month"]      = df["Date"].dt.month
df["Year"]       = df["Date"].dt.year
df["DayOfMonth"] = df["Date"].dt.day
df = df.join(store_meta_idx, on="Store")
df = df.sort_values(["Store","Date"]).reset_index(drop=True)

grp = df.groupby("Store")["Sales"]
df["Sales_Lag_1"]  = grp.shift(1)
df["Sales_Lag_7"]  = grp.shift(7)
df["Sales_Lag_14"] = grp.shift(14)
df["Sales_Rolling_Mean_7"]  = grp.shift(1).groupby(df["Store"]).transform(lambda x: x.rolling(7,  min_periods=1).mean())
df["Sales_Rolling_Mean_14"] = grp.shift(1).groupby(df["Store"]).transform(lambda x: x.rolling(14, min_periods=1).mean())

df = df.dropna(subset=["Sales_Lag_14"]).reset_index(drop=True)
print(f"Rows after dropping lag warmup NaNs: {len(df):,}")
df[FEATURES + ["Sales"]].head(3)

## 5. Train / Test Split

In [ ]:
split = pd.Timestamp(SPLIT_DATE)
train_df = df[df["Date"] < split]
test_df  = df[df["Date"] >= split]
print(f"Train: {len(train_df):,} rows  ({train_df['Date'].min().date()} to {train_df['Date'].max().date()})")
print(f"Test : {len(test_df):,}  rows  ({test_df['Date'].min().date()} to {test_df['Date'].max().date()})")

X_train, y_train = train_df[FEATURES], train_df["Sales"]
X_test,  y_test  = test_df[FEATURES],  test_df["Sales"]

## 6. Define Hyperparameter Variants

The baseline matches the deployed model. Three variants explore the impact of `num_leaves`, `learning_rate`, and `max_depth`.

In [ ]:
VARIANTS = [
    {
        "run_name": "lightgbm_store_baseline",
        "is_deployed": True,
        "params": {"n_estimators": 500, "learning_rate": 0.05, "num_leaves": 31, "max_depth": -1},
    },
    {
        "run_name": "lightgbm_store_more_leaves",
        "is_deployed": False,
        # More leaves = more complex trees, can capture finer store-level patterns
        "params": {"n_estimators": 500, "learning_rate": 0.05, "num_leaves": 63, "max_depth": -1},
    },
    {
        "run_name": "lightgbm_store_lower_lr",
        "is_deployed": False,
        # Lower learning rate requires more trees but often generalises better
        "params": {"n_estimators": 700, "learning_rate": 0.02, "num_leaves": 31, "max_depth": -1},
    },
    {
        "run_name": "lightgbm_store_deeper",
        "is_deployed": False,
        # Explicit depth cap to compare against unconstrained trees
        "params": {"n_estimators": 500, "learning_rate": 0.05, "num_leaves": 31, "max_depth": 8},
    },
]

def rmspe(y_true, y_pred):
    """Root Mean Squared Percentage Error -- the original Kaggle competition metric."""
    mask = y_true > 0
    return float(np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2)))

print(f"{len(VARIANTS)} variants defined.")

## 7. Train All Variants with MLflow Tracking

In [ ]:
run_ids = {}
results = []

for variant in VARIANTS:
    p = variant["params"]
    print(f"Training: {variant['run_name']} ...")

    model = LGBMRegressor(
        n_estimators=p["n_estimators"],
        learning_rate=p["learning_rate"],
        num_leaves=p["num_leaves"],
        max_depth=p["max_depth"],
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae      = mean_absolute_error(y_test, preds)
    rmspe_val = rmspe(y_test.values, preds)

    with mlflow.start_run(run_name=variant["run_name"]) as run:
        # Params
        for k, v in p.items():
            mlflow.log_param(k, v)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("split_date",   SPLIT_DATE)
        mlflow.log_param("features",     ",".join(FEATURES))
        mlflow.log_param("n_features",   len(FEATURES))
        mlflow.log_param("scope",        "per_store")
        mlflow.log_param("train_rows",   len(train_df))
        mlflow.log_param("test_rows",    len(test_df))

        # Metrics
        mlflow.log_metric("mae",   round(mae, 2))
        mlflow.log_metric("rmspe", round(rmspe_val, 4))

        # Tags
        mlflow.set_tag("model_type",  "ml")
        mlflow.set_tag("scope",       "per_store")
        mlflow.set_tag("notebook",    "06_store_level_forecasting")
        mlflow.set_tag("test_period", "2015-06-01_to_2015-07-31")

        if variant["is_deployed"]:
            mlflow.set_tag("deployed", "true")
            mlflow.set_tag("endpoint", "/forecast/store")

            # Feature importance plot (only for deployed baseline)
            FIG_DIR.mkdir(parents=True, exist_ok=True)
            importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
            fig, ax = plt.subplots(figsize=(10, 6))
            importance.plot(kind="bar", ax=ax, color="steelblue")
            ax.set_title("Feature Importance -- Per-Store LightGBM", fontsize=14)
            ax.set_ylabel("Importance (splits)")
            ax.set_xlabel("Feature")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            fig_path = FIG_DIR / "store_feature_importance.png"
            plt.savefig(fig_path, dpi=150)
            plt.show()

            mlflow.log_artifact(str(fig_path), "plots")

            # Log model artifact + save pkl for the API
            mlflow.lightgbm.log_model(model, "model")
            joblib.dump(model, MODEL_OUT)
            print(f"  Deployed model saved -> {MODEL_OUT}")

        run_ids[variant["run_name"]] = run.info.run_id
        results.append({"run_name": variant["run_name"], "mae": round(mae, 0), "rmspe_pct": round(rmspe_val * 100, 2), "run_id": run.info.run_id})
        print(f"  MAE={mae:,.0f}  RMSPE={rmspe_val*100:.2f}%  run_id={run.info.run_id}")

print("\nAll variants logged.")

## 8. Variant Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("mae").reset_index(drop=True)
print(results_df.to_string(index=False))
print("\nBest MAE variant:", results_df.iloc[0]["run_name"])
print("Deployed variant : lightgbm_store_baseline")

## 9. Feature Importance (Deployed Model)

In [ ]:
from PIL import Image
img = Image.open(FIG_DIR / "store_feature_importance.png")
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()

## 10. Run IDs Summary

In [ ]:
print("Experiment: rossmann-demand-forecasting")
print("-" * 75)
for name, rid in run_ids.items():
    deployed = " [DEPLOYED]" if name == "lightgbm_store_baseline" else ""
    print(f"  {name:45s}  {rid}{deployed}")

print("\nTo view in UI (run from project root):")
print("  mlflow ui --backend-store-uri sqlite:///mlflow.db")
print("  then open http://localhost:5000")